In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.option_price_discrepancy_performance AS


WITH tour_performance AS (
  SELECT 
    fb.supplier_id,
    fb.tour_id,
    fb.tour_option_id,
    dt.tour_title AS tour_title,
    dt.category AS tour_category,
    dt.activity_type AS activity_type,
    dtl.location_name AS location_name,
    
    -- LAST 3 MONTHS
    COUNT(DISTINCT CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 3 MONTH THEN fb.booking_id END) AS bookings_l3mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 3 MONTH THEN fb.gmv END) AS gmv_l3mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 3 MONTH THEN fb.nr END) AS nr_l3mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 3 MONTH THEN fb.tickets END) AS tickets_l3mo,
    COUNT(DISTINCT CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 3 MONTH THEN fb.customer_id_anon END) AS customers_l3mo,
    
    -- LAST 6 MONTHS
    COUNT(DISTINCT CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 6 MONTH THEN fb.booking_id END) AS bookings_l6mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 6 MONTH THEN fb.gmv END) AS gmv_l6mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 6 MONTH THEN fb.nr END) AS nr_l6mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 6 MONTH THEN fb.tickets END) AS tickets_l6mo,
    COUNT(DISTINCT CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 6 MONTH THEN fb.customer_id_anon END) AS customers_l6mo,
    
    -- LAST 12 MONTHS
    COUNT(DISTINCT CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH THEN fb.booking_id END) AS bookings_l12mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH THEN fb.gmv END) AS gmv_l12mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH THEN fb.nr END) AS nr_l12mo,
    SUM(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH THEN fb.tickets END) AS tickets_l12mo,
    COUNT(DISTINCT CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH THEN fb.customer_id_anon END) AS customers_l12mo,
    
    -- LIFETIME
    COUNT(DISTINCT fb.booking_id) AS bookings_lifetime,
    SUM(fb.gmv) AS gmv_lifetime,
    SUM(fb.nr) AS nr_lifetime,
    
    -- AVERAGE METRICS
    ROUND(AVG(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH THEN fb.gmv END), 2) AS avg_booking_value_l12mo,
    ROUND(AVG(CASE WHEN fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH THEN fb.tickets END), 1) AS avg_tickets_per_booking_l12mo,
    
    -- ENGAGEMENT SIGNALS
    MAX(fb.date_of_checkout) AS last_booking_date,
    MIN(fb.date_of_checkout) AS first_booking_date
    
  FROM production.dwh.fact_booking fb
  LEFT JOIN production.dwh.dim_tour dt
    ON fb.tour_id = dt.tour_id
  LEFT JOIN production.dwh.dim_tour_location dtl
    ON fb.location_id = dtl.location_id AND fb.tour_id = dtl.tour_id
  WHERE fb.date_of_checkout >= CURRENT_DATE - INTERVAL 12 MONTH
    AND fb.status_id = 1
  GROUP BY fb.supplier_id, fb.tour_id, fb.tour_option_id, dt.tour_title, dt.category, dt.activity_type, dtl.location_name
),

performance_with_percentiles AS (
  SELECT 
    *,
    
    -- PERCENTILE RANKINGS
    ROUND(PERCENT_RANK() OVER (ORDER BY gmv_l12mo) * 100, 1) AS gmv_percentile,
    ROUND(PERCENT_RANK() OVER (ORDER BY bookings_l12mo) * 100, 1) AS bookings_percentile,
    ROUND(PERCENT_RANK() OVER (ORDER BY nr_l12mo) * 100, 1) AS nr_percentile,
    ROUND(PERCENT_RANK() OVER (ORDER BY avg_booking_value_l12mo) * 100, 1) AS basket_size_percentile,
    
    -- NTILE BUCKETS
    NTILE(10) OVER (ORDER BY gmv_l12mo) AS gmv_decile,
    NTILE(4) OVER (ORDER BY gmv_l12mo) AS gmv_quartile
    
  FROM tour_performance
)

SELECT 
  supplier_id,
  tour_id,
  tour_option_id,
  tour_title,
  tour_category,
  activity_type,
  location_name,
  
  bookings_l3mo,
  gmv_l3mo,
  nr_l3mo,
  tickets_l3mo,
  customers_l3mo,
  
  bookings_l6mo,
  gmv_l6mo,
  nr_l6mo,
  tickets_l6mo,
  customers_l6mo,
  
  bookings_l12mo,
  gmv_l12mo,
  nr_l12mo,
  tickets_l12mo,
  customers_l12mo,
  
  bookings_lifetime,
  gmv_lifetime,
  nr_lifetime,
  
  avg_booking_value_l12mo,
  avg_tickets_per_booking_l12mo,
  
  last_booking_date,
  first_booking_date,
  
  gmv_percentile,
  bookings_percentile,
  nr_percentile,
  basket_size_percentile,
  
  gmv_decile,
  gmv_quartile,
  
  -- TIER BASED ON PERCENTILES
  CASE 
    WHEN gmv_l12mo IS NULL OR gmv_l12mo = 0 THEN 'no_revenue'
    WHEN gmv_percentile >= 95 THEN 'top_5pct'
    WHEN gmv_percentile >= 90 THEN 'top_10pct'
    WHEN gmv_percentile >= 75 THEN 'top_25pct'
    WHEN gmv_percentile >= 50 THEN 'above_median'
    ELSE 'below_median'
  END AS gmv_tier,
  
  -- SIMPLER TIER
  CASE 
    WHEN gmv_l12mo IS NULL OR gmv_l12mo = 0 THEN 'no_revenue'
    WHEN gmv_decile >= 9 THEN 'top_tier'
    WHEN gmv_decile >= 7 THEN 'high_value'
    WHEN gmv_decile >= 4 THEN 'medium_value'
    ELSE 'low_value'
  END AS tour_value_tier,
  
  -- DAYS SINCE LAST BOOKING
  DATEDIFF(CURRENT_DATE, last_booking_date) AS days_since_last_booking,
  
  -- GROWTH TREND
  CASE 
    WHEN bookings_l6mo > 0 
    THEN ROUND(100.0 * (bookings_l3mo * 2.0 - bookings_l6mo) / bookings_l6mo, 1)
    ELSE NULL
  END AS booking_growth_trend_pct

FROM performance_with_percentiles
ORDER BY gmv_l12mo DESC;
